# 03 - Softmax & CrossEntropyLoss

## 目标
- 手写 Softmax：把 logits 变成概率分布
- 手写 CrossEntropyLoss：衡量预测分布和真实分布的差距
- 理解数值稳定性：`x - np.max(x)` 防止 exp 爆炸
- 计算 loss 对 logits 的梯度

## 核心公式

### Softmax
$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

### Cross Entropy
$$
L = -\sum_i y_i \log p_i
$$

### 梯度（softmax + cross entropy 合并后的优雅结果）
$$
\frac{\partial L}{\partial z} = p - y
$$

In [ ]:
%run 02_linear_relu.ipynb

In [ ]:
class SoftmaxCrossEntropy:
    """合并 Softmax + CrossEntropyLoss，backward 自动简化为 p - y"""

    def forward(self, z, y_true):
        """
        z:      未归一化的 logits，shape (batch, 10)
        y_true: one-hot 标签，shape (batch, 10)
        返回:   loss（标量）
        """
        # 数值稳定性：每行减去最大值，防 exp 溢出
        # axis=1 沿着列方向，对每一行操作，保留维度，这样输出是列向量
        z_stable = z - np.max(z, axis=1, keepdims=True)

        # Softmax：exp → 归一化
        exp_z = np.exp(z_stable)
        self.probs = exp_z / np.sum(exp_z, axis=1, keepdims=True)  # (batch, 10)

        self.y_true = y_true        # 同名都没关系，俩个是不同的东西，把局部变量赋值给self对象属性
        batch_size = z.shape[0]

        # Cross Entropy: L = -sum(y * log(p)) / N
        # 加 1e-12 防 log(0)
        # 一般考虑的是平均loss，让loss与batchsize解耦，这样不同batchsize下loss也可以比较，同时梯度尺寸也与batchsize无关
        loss = -np.sum(y_true * np.log(self.probs + 1e-12)) / batch_size

        return loss

    def backward(self):
        """返回 dL/dz，即损失对 logits（在Softmax之前的原始分数） 的梯度"""
        batch_size = self.y_true.shape[0]
        return (self.probs - self.y_true) / batch_size  # (batch, 10)

In [ ]:
# # 组装一个小型前向通路测试
# linear = Linear(784, 128)
# linear2 = Linear(128, 10)    # 第二层：128 → 10，输出 logits
# relu = ReLU()
# loss_fn = SoftmaxCrossEntropy()

# # 取 3 个样本
# batch_x = x_train[:3]          # (3, 784)
# batch_y = y_train[:3]          # (3, 10)

# # forward
# z1 = linear.forward(batch_x)   # (3, 128)
# a1 = relu.forward(z1)          # (3, 128)
# z2 = linear2.forward(a1)       # (3, 10)  ← logits
# loss = loss_fn.forward(z2, batch_y)

# print(f"z2 shape: {z2.shape}")
# print(f"预测概率:\n{loss_fn.probs}")     # 每行和 = 1
# print(f"loss: {loss:.4f}")

# # backward
# dz2 = loss_fn.backward()        # (3, 10)，这就是 ∂L/∂z = p - y
# da1 = linear2.backward(dz2)     # (3, 128)
# dz1 = relu.backward(da1)        # (3, 128)
# dx = linear.backward(dz1)       # (3, 784)

# print(f"\ndz2 shape: {dz2.shape}")       # (3, 10)
# print(f"dx shape:  {dx.shape}")          # (3, 784)
# print(f"\ndz2 (前 3 行):\n{dz2[:3]}")     # 梯度 p-y，值应该是小数